In [1]:
# app.py
import streamlit as st
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama

# Function to process PDF and return retriever + LLM
def load_pdf_and_create_rag(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = text_splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vector_db = Chroma.from_documents(chunks, embeddings)

    retriever = vector_db.as_retriever(search_kwargs={"k": 3})

    llm = Ollama(model="llama3")

    return retriever, llm

In [2]:
st.title("📄 Chat with Your PDF (Local LLM)")

uploaded_file = st.file_uploader("Upload a PDF", type="pdf")

if uploaded_file:
    # Save the uploaded PDF temporarily
    with open("temp.pdf", "wb") as f:
        f.write(uploaded_file.read())

    # Load RAG system
    retriever, llm = load_pdf_and_create_rag("temp.pdf")

    # Input box for question
    question = st.text_input("Ask a question about the document")

    if question:
        docs = retriever.invoke(question)
        context = "\n\n".join([doc.page_content for doc in docs])

        prompt = f"""
        Answer the question using the context below.

        Context:
        {context}

        Question:
        {question}
        """

        response = llm.invoke(prompt)

        st.write("### Answer")
        st.write(response)

2026-03-21 19:06:29.199 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 19:06:29.507 
  command:

    streamlit run C:\Users\iment\Desktop\LLM_PROJECT\env\lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-03-21 19:06:29.507 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 19:06:29.507 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 19:06:29.518 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 19:06:29.519 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 19:06:29.520 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 19:06:29.522 Thread 'M